# Vortex-AI Dynamic Graph Regime & Risk Notebook

This notebook downloads NIFTY-50 prices, preprocesses them into 60-day return windows
with crisis-regime labels, builds empirical (Pearson + Granger) adjacency graphs, trains a
Dynamic Graph Attention Network (GAT) for regime classification, and produces an EDA
dashboard. It runs on Kaggle (`/kaggle/working`) or a local checkout (`data/raw`).

In [23]:
# Vortex-AI notebook setup.
# Resolves the raw-data directory for both Kaggle (/kaggle/working) and a local checkout.
%pip install yfinance
from pathlib import Path

def resolve_raw_data_dir() -> Path:
    kaggle_dir = Path("/kaggle/working/data/raw")
    if kaggle_dir.parent.exists():
        return kaggle_dir
    local_dir = Path.cwd() / "data" / "raw"
    local_dir.mkdir(parents=True, exist_ok=True)
    return local_dir

RAW_DATA_DIR = resolve_raw_data_dir()
print(f"Raw data directory: {RAW_DATA_DIR}")

Note: you may need to restart the kernel to use updated packages.
Raw data directory: \kaggle\working\data\raw



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
import numpy as np
import pandas as pd
import yfinance as yf

# 50 largest NIFTY constituents (Yahoo Finance uses the ".NS" suffix for NSE tickers).
NIFTY50 = [
    "RELIANCE.NS", "TCS.NS", "HDFCBANK.NS", "INFY.NS", "HINDUNILVR.NS",
    "ICICIBANK.NS", "KOTAKBANK.NS", "BHARTIARTL.NS", "ITC.NS", "AXISBANK.NS",
    "SBIN.NS", "LT.NS", "BAJFINANCE.NS", "HCLTECH.NS", "ASIANPAINT.NS",
    "MARUTI.NS", "SUNPHARMA.NS", "TITAN.NS", "ULTRACEMCO.NS", "NESTLEIND.NS",
    "WIPRO.NS", "POWERGRID.NS", "NTPC.NS", "M&M.NS", "TECHM.NS",
    "TATAMOTORS.NS", "TATASTEEL.NS", "JSWSTEEL.NS", "BAJAJ-AUTO.NS", "CIPLA.NS",
    "DRREDDY.NS", "DIVISLAB.NS", "HEROMOTOCO.NS", "ONGC.NS", "COALINDIA.NS",
    "BPCL.NS", "GRASIM.NS", "ADANIPORTS.NS", "EICHERMOT.NS", "APOLLOHOSP.NS",
    "HINDALCO.NS", "TATACONSUM.NS", "BRITANNIA.NS", "SHREECEM.NS", "UPL.NS",
    "BAJAJFINSV.NS", "SBILIFE.NS", "HDFCLIFE.NS", "INDUSINDBK.NS", "LTIM.NS",
]

print("Downloading NIFTY 50 historical data...")
prices = yf.download(NIFTY50, start="2010-01-01", end="2024-12-31", auto_adjust=True, threads=True)
close = prices["Close"].ffill().bfill()
volume = prices["Volume"].ffill().bfill()

close.to_csv(RAW_DATA_DIR / "nifty50_close.csv")
volume.to_csv(RAW_DATA_DIR / "nifty50_volume.csv")
print(f"Download complete: {close.shape[0]} trading days x {close.shape[1]} stocks.")

[                       0%                       ]

[*******************   40%                       ]  20 of 50 completed$TATAMOTORS.NS: possibly delisted; no timezone found
[**********************76%***********            ]  38 of 50 completed$LTIM.NS: possibly delisted; no timezone found
[*********************100%***********************]  50 of 50 completed

2 Failed downloads:
['TATAMOTORS.NS', 'LTIM.NS']: possibly delisted; no timezone found


Download complete: 3700 trading days x 50 stocks.


In [25]:
import numpy as np
import pandas as pd

def compute_log_returns(close_df: pd.DataFrame) -> pd.DataFrame:
    # Coerce to numeric, interpolate gaps, then compute daily log returns.
    close_df = close_df.apply(pd.to_numeric, errors="coerce").ffill().bfill()
    returns = np.log(close_df / close_df.shift(1))
    return returns.fillna(0.0)

def label_regimes(returns_df: pd.DataFrame, vol_window: int = 20, crisis_pct: float = 85.0) -> pd.Series:
    # Composite crisis score: realized vol + 5d drawdown fraction + circuit-breaker proximity.
    realized_vol = returns_df.std(axis=1).rolling(vol_window, min_periods=1).mean()
    rolling_5d = returns_df.rolling(5, min_periods=1).sum()
    drawdown_frac = (rolling_5d < -0.05).mean(axis=1)
    circuit_prox = (returns_df.abs() > 0.095).mean(axis=1)

    vol_q99 = realized_vol.quantile(0.99)
    vol_norm = realized_vol / vol_q99 if (not pd.isna(vol_q99) and vol_q99 > 0) else realized_vol

    crisis_score = (0.5 * vol_norm + 0.3 * drawdown_frac + 0.2 * circuit_prox).fillna(0.0)
    threshold = np.percentile(crisis_score, crisis_pct)
    return (crisis_score >= threshold).astype(int)

def make_windows(returns_df: pd.DataFrame, labels: pd.Series, T: int = 60):
    # Slide a window of T trading days; label each window by its final-day regime.
    windows, window_labels = [], []
    arr, lab = returns_df.values, labels.values
    for i in range(T, len(arr)):
        windows.append(arr[i - T : i])
        window_labels.append(lab[i - 1])
    return np.array(windows, dtype=np.float32), np.array(window_labels, dtype=np.int64)

close_df = pd.read_csv(RAW_DATA_DIR / "nifty50_close.csv", index_col=0, parse_dates=True)
returns_df = compute_log_returns(close_df)
labels = label_regimes(returns_df, vol_window=20, crisis_pct=85.0)
windows, window_labels = make_windows(returns_df, labels, T=60)

np.save(RAW_DATA_DIR / "returns.npy", returns_df.values)
np.save(RAW_DATA_DIR / "regimes.npy", labels.values)
np.save(RAW_DATA_DIR / "windows.npy", windows)
np.save(RAW_DATA_DIR / "window_regimes.npy", window_labels)

print(f"Preprocessed: {returns_df.shape[0]} Days | Crisis Days: {labels.sum()} ({labels.mean():.1%})")
print(f"Created 60-Day Windows Tensor: {windows.shape}")

Preprocessed: 3700 Days | Crisis Days: 555 (15.0%)
Created 60-Day Windows Tensor: (3640, 60, 50)


In [26]:
from pathlib import Path
import time
import numpy as np
import torch
from scipy.stats import f as f_distribution

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Building empirical graphs on device: {device}")

# Build adjacency as alpha * Pearson(correlation) + (1 - alpha) * Granger(causality).
# - Pearson captures contemporaneous co-movement (symmetric).
# - Granger captures directional lead-lag predictability (asymmetric F-test).
def build_empirical_graphs(
    returns_tensor: torch.Tensor,
    window_size: int = 60,
    granger_lag: int = 2,
    pearson_threshold: float = 0.3,
    granger_alpha: float = 0.5,
    batch_size: int = 256,
):
    num_days, num_assets = returns_tensor.shape
    # Match the preprocessing window count exactly (start indices 0 .. num_days - T - 1).
    num_windows = num_days - window_size
    windows = (
        returns_tensor.unfold(0, window_size, 1).permute(0, 2, 1)[:num_windows].to(device)
    )  # [W, T, N]

    # 1. Batched Pearson correlation per window (vectorised over assets).
    centered = windows - windows.mean(dim=1, keepdim=True)
    covariance = torch.bmm(centered.transpose(1, 2), centered) / (window_size - 1)
    std = torch.sqrt(torch.diagonal(covariance, dim1=1, dim2=2).clamp(min=1e-8)).unsqueeze(-1)
    correlation = covariance / (std @ std.transpose(1, 2))
    correlation = torch.nan_to_num(correlation, nan=0.0)
    pearson_adj = torch.where(correlation.abs() >= pearson_threshold, correlation.abs(), 0.0)
    eye_mask = torch.eye(num_assets, device=device, dtype=torch.bool)
    pearson_adj[:, eye_mask] = 0.0

    # 2. Batched Granger causality F-test for every ordered pair (i -> j).
    m_obs = window_size - granger_lag
    df_num = float(granger_lag)
    df_den = float(m_obs - (2 * granger_lag + 1))

    targets = windows[:, granger_lag:, :].transpose(1, 2).unsqueeze(-1)  # [W, N, M, 1]
    ones = torch.ones((num_windows, num_assets, m_obs, 1), device=device)
    lag_1 = windows[:, granger_lag - 1 : window_size - 1, :].transpose(1, 2).unsqueeze(-1)
    lag_2 = windows[:, granger_lag - 2 : window_size - 2, :].transpose(1, 2).unsqueeze(-1)
    restricted_design = torch.cat([ones, lag_1, lag_2], dim=-1)  # [W, N, M, 3]

    # Restricted regression (target only on its own lags) -> RSS_r used by every pair.
    a_restricted = restricted_design.transpose(-2, -1) @ restricted_design + 1e-6 * torch.eye(3, device=device)
    b_restricted = restricted_design.transpose(-2, -1) @ targets
    coef_restricted = torch.linalg.pinv(a_restricted) @ b_restricted
    target_energy = (targets ** 2).sum(dim=-2)  # [W, N, 1]
    rss_restricted = target_energy - (b_restricted.transpose(-2, -1) @ coef_restricted).squeeze(-1)

    granger_batches = []
    for start in range(0, num_windows, batch_size):
        end = min(start + batch_size, num_windows)
        b_w = end - start

        # Unrestricted design adds the candidate predictor's lags for each (i, j) pair.
        batch_targets = targets[start:end]
        batch_design = restricted_design[start:end].unsqueeze(2).expand(b_w, num_assets, num_assets, m_obs, 3)
        pred_lag_1 = windows[start:end, granger_lag - 1 : window_size - 1, :].transpose(1, 2).unsqueeze(1).unsqueeze(-1)
        pred_lag_2 = windows[start:end, granger_lag - 2 : window_size - 2, :].transpose(1, 2).unsqueeze(1).unsqueeze(-1)
        predictor_lags = torch.cat([pred_lag_1, pred_lag_2], dim=-1).expand(b_w, num_assets, num_assets, m_obs, 2)
        unrestricted_design = torch.cat([batch_design, predictor_lags], dim=-1)  # [b, N, N, M, 5]

        a_unrestricted = unrestricted_design.transpose(-2, -1) @ unrestricted_design + 1e-6 * torch.eye(5, device=device)
        b_unrestricted = unrestricted_design.transpose(-2, -1) @ batch_targets.unsqueeze(2)
        coef_unrestricted = torch.linalg.pinv(a_unrestricted) @ b_unrestricted

        batch_energy = target_energy[start:end].unsqueeze(2).unsqueeze(-1)
        rss_unrestricted = (batch_energy - b_unrestricted.transpose(-2, -1) @ coef_unrestricted).squeeze(-1).squeeze(-1)

        batch_rss_restricted = rss_restricted[start:end].expand(b_w, num_assets, num_assets)
        f_statistic = ((batch_rss_restricted - rss_unrestricted) / df_num) / (rss_unrestricted / df_den)
        f_statistic = f_statistic.clamp(min=0.0)

        p_values = torch.from_numpy(
            f_distribution.sf(f_statistic.cpu().numpy(), df_num, df_den)
        ).to(device).float()
        batch_granger = torch.where(p_values < 0.05, 1.0 - p_values, 0.0)
        batch_granger[:, eye_mask] = 0.0
        granger_batches.append(batch_granger)

    granger_adj = torch.cat(granger_batches, dim=0)
    return granger_alpha * pearson_adj + (1.0 - granger_alpha) * granger_adj

start_time = time.time()
returns_path = RAW_DATA_DIR / "returns.npy"
if not returns_path.exists():
    raise FileNotFoundError(f"Missing {returns_path}. Run the preprocessing cell first!")

returns_tensor = torch.from_numpy(np.load(returns_path)).float()
adjacency_matrices = build_empirical_graphs(returns_tensor, window_size=60, granger_alpha=0.5, batch_size=256)

output_path = RAW_DATA_DIR / "adj_matrices.npy"
np.save(output_path, adjacency_matrices.cpu().numpy())

elapsed = time.time() - start_time
print(f"Finished in {elapsed:.2f} seconds!")
print(f"Saved adjacency tensor to {output_path}: {adjacency_matrices.shape}")

Building empirical graphs on device: cuda
Finished in 56.01 seconds!
Saved adjacency tensor to \kaggle\working\data\raw\adj_matrices.npy: torch.Size([3640, 50, 50])


In [27]:
FILES = [
    "returns.npy",
    "regimes.npy",
    "windows.npy",
    "window_regimes.npy",
    "adj_matrices.npy",
]

print("DATA VERIFICATION")
for fname in FILES:
    path = RAW_DATA_DIR / fname
    if path.exists():
        arr = np.load(path)
        print(f"{fname:<20} | Shape: {str(arr.shape):<18} | Type: {arr.dtype}")
    else:
        print(f"Missing: {fname}")

DATA VERIFICATION
returns.npy          | Shape: (3700, 50)         | Type: float64
regimes.npy          | Shape: (3700,)            | Type: int64
windows.npy          | Shape: (3640, 60, 50)     | Type: float32
window_regimes.npy   | Shape: (3640,)            | Type: int64
adj_matrices.npy     | Shape: (3640, 50, 50)     | Type: float32


In [28]:
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_count = torch.cuda.device_count()
use_amp = device.type == "cuda"
print(f"Training on device: {device} | GPUs: {gpu_count} | Mixed precision: {use_amp}")

class VortexGraphDataset(Dataset):
    # Windowed NIFTY-50 graphs with crisis-regime labels for supervised training.
    def __init__(self, data_dir: Path):
        windows = np.load(data_dir / "windows.npy")          # [num_windows, T, N]
        adjacency = np.load(data_dir / "adj_matrices.npy")    # [num_windows, N, N]
        labels = np.load(data_dir / "window_regimes.npy")     # [num_windows]

        # Defensive alignment (counts should already match after preprocessing).
        num_samples = min(len(windows), len(adjacency), len(labels))
        windows, adjacency, labels = windows[:num_samples], adjacency[:num_samples], labels[:num_samples]

        # Node features = per-stock lookback returns -> shape [N, T].
        self.node_features = torch.from_numpy(windows).permute(0, 2, 1).float()
        self.adjacency = torch.from_numpy(adjacency).float()
        self.labels = torch.from_numpy(labels).long()
        print(f"Dataset: {len(self.labels)} samples | Nodes: {self.node_features.shape[1]} | Lookback T: {self.node_features.shape[2]}")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.node_features[idx], self.adjacency[idx], self.labels[idx]

class GraphAttentionLayer(nn.Module):
    # Single graph-attention layer that masks attention using the empirical adjacency.
    def __init__(self, in_features: int, out_features: int, dropout: float = 0.2):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features, bias=False)
        self.attention = nn.Linear(2 * out_features, 1, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.leaky_relu = nn.LeakyReLU(0.2)

    def forward(self, x: torch.Tensor, adjacency: torch.Tensor) -> torch.Tensor:
        # x: [B, N, in_features], adjacency: [B, N, N]
        batch_size, num_nodes, _ = x.shape
        projected = self.linear(x)  # [B, N, out_features]

        source = projected.unsqueeze(2).expand(batch_size, num_nodes, num_nodes, -1)
        target = projected.unsqueeze(1).expand(batch_size, num_nodes, num_nodes, -1)
        pair_features = torch.cat([source, target], dim=-1)

        scores = self.leaky_relu(self.attention(pair_features)).squeeze(-1)  # [B, N, N]
        negative_inf = -9e15 * torch.ones_like(scores)
        masked_scores = torch.where(adjacency > 0, scores, negative_inf)
        attention_weights = self.dropout(F.softmax(masked_scores, dim=-1))

        return F.elu(attention_weights @ projected)

class DynamicGATEncoder(nn.Module):
    # Two GAT layers that pool node embeddings into a regime classification logit.
    def __init__(self, in_dim: int = 60, hidden_dim: int = 64, out_dim: int = 32, num_classes: int = 2):
        super().__init__()
        self.gat1 = GraphAttentionLayer(in_dim, hidden_dim)
        self.gat2 = GraphAttentionLayer(hidden_dim, out_dim)
        self.classifier = nn.Sequential(
            nn.Linear(out_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

    def forward(self, x: torch.Tensor, adjacency: torch.Tensor) -> torch.Tensor:
        hidden = self.gat2(self.gat1(x, adjacency))
        graph_embedding = torch.cat([hidden.mean(dim=1), hidden.max(dim=1).values], dim=-1)
        return self.classifier(graph_embedding)

# Load data and split CHRONOLOGICALLY to avoid temporal leakage (train past, validate future).
full_dataset = VortexGraphDataset(RAW_DATA_DIR)
num_samples = len(full_dataset)
train_size = int(0.8 * num_samples)
train_dataset = Subset(full_dataset, range(0, train_size))
val_dataset = Subset(full_dataset, range(train_size, num_samples))

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, pin_memory=use_amp, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, pin_memory=use_amp, num_workers=0)

model = DynamicGATEncoder(in_dim=60, hidden_dim=64, out_dim=32, num_classes=2)
if gpu_count > 1:  # nn.DataParallel is legacy; prefer DistributedDataParallel for multi-GPU.
    model = nn.DataParallel(model)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scaler = torch.amp.GradScaler(enabled=use_amp)
epochs = 20
print("Starting Dynamic GAT training...")

def run_epoch(model, loader, training: bool):
    model.train(training)
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for features_b, adjacency_b, labels_b in loader:
            features_b = features_b.to(device, non_blocking=True)
            adjacency_b = adjacency_b.to(device, non_blocking=True)
            labels_b = labels_b.to(device, non_blocking=True)

            if training:
                optimizer.zero_grad()
            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                logits = model(features_b, adjacency_b)
                loss = criterion(logits, labels_b)
            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            total_loss += loss.item() * features_b.size(0)
            correct += (logits.argmax(dim=-1) == labels_b).sum().item()
            total += features_b.size(0)
    return total_loss / total, correct / total

for epoch in range(1, epochs + 1):
    train_loss, train_acc = run_epoch(model, train_loader, training=True)
    val_loss, val_acc = run_epoch(model, val_loader, training=False)
    print(f"Epoch [{epoch:02d}/{epochs}] | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2%} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2%}")

checkpoint_path = RAW_DATA_DIR.parent / "gat_encoder.pth"
torch.save(model.module.state_dict() if gpu_count > 1 else model.state_dict(), checkpoint_path)
print(f"Training complete! Encoder saved to {checkpoint_path}")

Training on device: cuda | GPUs: 1 | Mixed precision: True
Dataset: 3640 samples | Nodes: 50 | Lookback T: 60
Starting Dynamic GAT training...


RuntimeError: DataLoader worker (pid(s) 27828, 19316) exited unexpectedly

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RAW_DIR = Path(RAW_DATA_DIR)
PLOT_DIR = RAW_DIR.parent / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-darkgrid" if "seaborn-v0_8-darkgrid" in plt.style.available else "default")
plt.rcParams["figure.dpi"] = 300
plt.rcParams["font.size"] = 10

returns = np.load(RAW_DIR / "returns.npy")
regimes = np.load(RAW_DIR / "regimes.npy")
adjacency = np.load(RAW_DIR / "adj_matrices.npy")
window_regimes = np.load(RAW_DIR / "window_regimes.npy")

num_windows = min(len(adjacency), len(window_regimes))
adjacency, window_regimes = adjacency[:num_windows], window_regimes[:num_windows]

try:
    close_df = pd.read_csv(RAW_DIR / "nifty50_close.csv", index_col=0, parse_dates=True)
    tickers = [c.replace(".NS", "") for c in close_df.columns]
    dates = close_df.index[-len(returns):]
except Exception:
    tickers = [f"S{i:02d}" for i in range(returns.shape[1])]
    dates = np.arange(len(returns))

window_dates = dates[60 : 60 + num_windows]
num_assets = returns.shape[1]
max_edges = num_assets * (num_assets - 1)

fig, axes = plt.subplots(3, 2, figsize=(18, 15))
plt.subplots_adjust(hspace=0.35, wspace=0.25)

# Panel 1: cumulative return with crisis shading.
ax1 = axes[0, 0]
cumulative = np.cumsum(returns.mean(axis=1))
ax1.plot(dates, cumulative, color="#1f77b4", linewidth=1.2, label="NIFTY-50 Log Return")
ax1.fill_between(dates, cumulative.min(), cumulative.max(), where=(regimes[: len(dates)] == 1), color="crimson", alpha=0.35, label="Crisis Regime (~15% Stress)")
ax1.set_title("1. NIFTY-50 Cumulative Returns & Identified Crisis Regimes", fontweight="bold")
ax1.set_ylabel("Cumulative Log Return")
ax1.legend(loc="upper left")

# Panel 2: return density by regime (crisis shows fatter tails).
ax2 = axes[0, 1]
normal_ret = returns[regimes == 0].flatten()
crisis_ret = returns[regimes == 1].flatten()
sns.kdeplot(normal_ret, ax=ax2, color="royalblue", label="Normal Regime", fill=True, alpha=0.25)
sns.kdeplot(crisis_ret, ax=ax2, color="crimson", label="Crisis Regime", fill=True, alpha=0.25)
ax2.set_xlim(-0.08, 0.08)
ax2.set_title("2. Return Distribution (Fat-Tail Spikes in Crisis)", fontweight="bold")
ax2.set_xlabel("Daily Log Return")
ax2.set_ylabel("Density")
ax2.legend()

# Panel 3: dynamic graph edge density over time.
ax3 = axes[1, 0]
edge_density = [np.count_nonzero(matrix) / max_edges for matrix in adjacency]
ax3.plot(window_dates, edge_density, color="purple", linewidth=1.0, label="Graph Density")
ax3.fill_between(window_dates, min(edge_density), max(edge_density), where=(window_regimes == 1), color="crimson", alpha=0.2, label="Crisis Window")
ax3.set_title("3. Dynamic Graph Edge Density Over Time (T=60 Windows)", fontweight="bold")
ax3.set_ylabel("Density (Edges / Max Possible)")
ax3.legend(loc="upper left")

# Panel 4: systemic risk via spectral radius (largest |eigenvalue| of each adjacency).
ax4 = axes[1, 1]
spectral_radius = [np.max(np.abs(np.linalg.eigvals(matrix))) for matrix in adjacency]
ax4.plot(window_dates, spectral_radius, color="darkorange", linewidth=1.0, label=r"Max Eigenvalue $\lambda_{max}$")
ax4.fill_between(window_dates, min(spectral_radius), max(spectral_radius), where=(window_regimes == 1), color="crimson", alpha=0.2, label="Crisis Window")
ax4.set_title(r"4. Systemic Risk Trajectory ($\lambda_{max}$ Spectral Radius)", fontweight="bold")
ax4.set_ylabel(r"Spectral Radius $\lambda_{max}$")
ax4.legend(loc="upper left")

# Panel 5: mean adjacency in the normal regime.
ax5 = axes[2, 0]
mean_normal = adjacency[window_regimes == 0].mean(axis=0)
sns.heatmap(mean_normal, ax=ax5, cmap="Blues", cbar=True, xticklabels=False, yticklabels=False)
ax5.set_title(r"5. Mean Adjacency Matrix $ar{A}_{normal}$", fontweight="bold")
ax5.set_xlabel("Stock Index (0-49)")
ax5.set_ylabel("Stock Index (0-49)")

# Panel 6: mean adjacency in the crisis regime (tighter coupling).
ax6 = axes[2, 1]
mean_crisis = adjacency[window_regimes == 1].mean(axis=0)
sns.heatmap(mean_crisis, ax=ax6, cmap="Reds", cbar=True, xticklabels=False, yticklabels=False)
ax6.set_title(r"6. Mean Adjacency Matrix $ar{A}_{crisis}$ (Tight Coupling)", fontweight="bold")
ax6.set_xlabel("Stock Index (0-49)")
ax6.set_ylabel("Stock Index (0-49)")

dashboard_path = PLOT_DIR / "vortex_master_eda_dashboard.png"
plt.savefig(dashboard_path, bbox_inches="tight")
plt.show()
print(f"Saved Master EDA Dashboard to: {dashboard_path}")

# Network hub analysis: top out-degree central stocks per regime.
fig_hub, (ax_hub1, ax_hub2) = plt.subplots(1, 2, figsize=(16, 6))
degree_normal = mean_normal.sum(axis=1)
degree_crisis = mean_crisis.sum(axis=1)
top_normal = np.argsort(degree_normal)[-10:]
top_crisis = np.argsort(degree_crisis)[-10:]

ax_hub1.barh([tickers[i] for i in top_normal], degree_normal[top_normal], color="royalblue")
ax_hub1.set_title("Top 10 Central Stocks (Normal Regime)", fontweight="bold")
ax_hub1.set_xlabel("Total Out-Degree Centrality Weight")

ax_hub2.barh([tickers[i] for i in top_crisis], degree_crisis[top_crisis], color="crimson")
ax_hub2.set_title("Top 10 Central Stocks (Crisis Regime - Systemic Risk Drivers)", fontweight="bold")
ax_hub2.set_xlabel("Total Out-Degree Centrality Weight")

plt.tight_layout()
hubs_path = PLOT_DIR / "vortex_network_hubs.png"
plt.savefig(hubs_path, bbox_inches="tight")
plt.show()
print(f"Saved Network Hub Analysis to: {hubs_path}")